# OpenCV Document Validator — Results Explorer

**Two modes:**
1. **Load existing results** — reads `eval_results/batch_results.csv`, re-runs OpenCV to enrich with raw feature scores
2. **Re-run on folder** — point `SCAN_FOLDER` to any directory, run OpenCV fresh, get full score dataframe

Change thresholds in Section 1 and re-run any section independently.

## 1. Config — thresholds

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / "src"))

import dataclasses
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2
from IPython.display import display
from sklearn.metrics import classification_report, confusion_matrix, f1_score

from document_validation import ValidationConfig, validate_document_file_ensemble
from document_validation.validator import validate_document_image, _load_image, _render_pdf_pages
from document_validation.ground_truth import FOLDER_TO_LABEL

# ── Thresholds ──────────────────────────────────────────────────────────────
# Objective: reject only with HIGH confidence.
# False negatives (bad docs accepted) are OK — downstream validator catches them.
# All blur decisions use Tenengrad (Sobel gradient magnitude) — more robust than Laplacian.
# Key parameters:
#   blur_tenengrad_threshold : ≥ this = sharp; below = blurry
#   blur_patch_ratio         : watermark-robust patch check gate (ratio × threshold)
#   blur_patch_grid          : 5×5 grid; 10th-pct of non-uniform patches
#   min_document_confidence  : 0.60 + content fallback (edge/ink present)
#   min_reject_confidence    : 0.75 gate — only reject when detector is ≥75% confident
CONFIG = ValidationConfig(
    blur_tenengrad_threshold     = 60.0,
    blur_patch_ratio             = 0.25,
    blur_patch_grid              = 5,
    blur_patch_percentile        = 10.0,
    min_readability_contrast     = 35.0,
    max_low_readability_gray_std = 65.0,
    min_document_confidence      = 0.60,
    min_reject_confidence        = 0.75,
    min_cut_confidence           = 0.85,
    pdf_dpi                      = 200,
)

threshold_fields = [
    "blur_tenengrad_threshold", "blur_patch_ratio",
    "blur_patch_grid", "blur_patch_percentile",
    "min_readability_contrast", "max_low_readability_gray_std",
    "min_document_confidence", "min_reject_confidence",
    "min_cut_confidence", "pdf_dpi",
]
cfg_dict = dataclasses.asdict(CONFIG)
display(pd.DataFrame(
    [(k, cfg_dict[k]) for k in threshold_fields],
    columns=["threshold", "value"]
).set_index("threshold"))

PROJECT_ROOT = Path.cwd()
OUT_DIR = PROJECT_ROOT / "eval_results"
OUT_DIR.mkdir(exist_ok=True)
EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".pdf"}
LABELS = ["accepted", "blur", "cut", "not_document"]
TIE_BREAK = ("not_document", "blur", "not_clear", "cut", "accepted")


## 2. Helpers — score extraction

In [2]:
def _label_from_issues(issues):
    normalised = ['blur' if i == 'not_clear' else i for i in issues]
    if not normalised:
        return 'accepted'
    for label in TIE_BREAK:
        if label in normalised:
            return label
    return normalised[0]


def score_page(page_bgr, config=CONFIG):
    """Run OpenCV on one BGR image, return flat score dict."""
    r = validate_document_image(page_bgr, config=config)
    return {
        # blur
        'laplacian_variance'  : round(r.blur.laplacian_variance, 2),
        'tenengrad'           : round(r.blur.tenengrad, 2),
        'readability_contrast': round(r.blur.readability_contrast, 2),
        'grayscale_std'       : round(r.blur.grayscale_std, 2),
        'blur_confidence'     : round(r.blur.confidence, 3),
        # document
        'doc_confidence'      : round(r.document.confidence, 3),
        'page_area_ratio'     : round(r.document.page_area_ratio, 3),
        'ink_ratio'           : round(r.document.ink_ratio, 3),
        'edge_density'        : round(r.document.edge_density, 3),
        # cut
        'cut_confidence'      : round(r.cut.confidence, 3),
        # clear
        'clear_confidence'    : round(r.clear.confidence, 3),
        # thresholds used
        'thresh_laplacian'    : config.blur_laplacian_threshold,
        'thresh_tenengrad'    : config.blur_tenengrad_threshold,
        'thresh_min_doc_conf' : config.min_document_confidence,
        'thresh_min_reject'   : config.min_reject_confidence,
        # prediction
        'raw_issues'          : ', '.join(r.issues) if r.issues else '',
        'predicted_label'     : _label_from_issues(r.issues),
    }


def score_file(file_path, config=CONFIG):
    """Score the first page of a file, return score dict."""
    fp = Path(file_path)
    try:
        if fp.suffix.lower() == '.pdf':
            pages = list(_render_pdf_pages(fp, config))
            page = pages[0] if pages else None
        else:
            page = _load_image(fp)
        if page is None:
            raise ValueError('empty page')
        scores = score_page(page, config)
    except Exception as exc:
        scores = {'predicted_label': 'error', 'raw_issues': str(exc)}
    scores['file_path'] = str(fp)
    scores['filename']  = fp.name
    return scores


print('Helpers loaded.')

Helpers loaded.


## 3. Load existing results

Reads `eval_results/batch_results.csv` (319 files from the last full eval run),
then re-scores each file with the current `CONFIG` to attach raw feature columns.

In [ ]:
BATCH_CSV = OUT_DIR / 'batch_results.csv'

if not BATCH_CSV.exists():
    print(f'batch_results.csv not found at {BATCH_CSV}. Run run_eval.py first.')
else:
    base_df = pd.read_csv(BATCH_CSV)
    base_df['file_path'] = base_df['file_path'].apply(Path)
    print(f'Loaded {len(base_df)} rows from {BATCH_CSV}')
    display(base_df['expected_label'].value_counts().rename('count').to_frame())

In [ ]:
# Re-score with current thresholds to get raw feature columns
# This takes ~5-8 minutes (same as run_eval.py)
# Set RESCORE = False to skip and use only what's in the CSV
RESCORE = True

if RESCORE and BATCH_CSV.exists():
    score_rows = []
    n = len(base_df)
    for i, (_, row) in enumerate(base_df.iterrows()):
        s = score_file(row['file_path'], CONFIG)
        s['expected_label'] = row['expected_label']
        s['doc_type']       = row.get('doc_type', '')
        s['state']          = row.get('state', '')
        score_rows.append(s)
        if (i + 1) % 50 == 0 or i == n - 1:
            print(f'  {i+1}/{n}', end='\r')

    results_df = pd.DataFrame(score_rows)
    results_df.to_csv(OUT_DIR / 'batch_results_scored.csv', index=False)
    print(f'\nRe-scored {len(results_df)} files → eval_results/batch_results_scored.csv')
else:
    # Load from pre-scored CSV if it exists
    scored_csv = OUT_DIR / 'batch_results_scored.csv'
    if scored_csv.exists():
        results_df = pd.read_csv(scored_csv)
        print(f'Loaded pre-scored results: {len(results_df)} rows')
    else:
        results_df = base_df.copy()
        print('Using base CSV (no feature scores — set RESCORE=True to enrich)')

In [3]:
col_order = [
    'filename', 'doc_type', 'state', 'expected_label', 'predicted_label', 'raw_issues',
    'watermark_blur',
    'laplacian_variance', 'tenengrad', 'readability_contrast', 'grayscale_std',
    'blur_confidence', 'doc_confidence', 'cut_confidence', 'clear_confidence',
    'ink_ratio', 'edge_density', 'page_area_ratio',
    'thresh_laplacian', 'thresh_tenengrad', 'thresh_min_doc_conf', 'thresh_min_reject',
    'file_path',
]
def highlight_wrong(row):
    wrong = row.get('expected_label') != row.get('predicted_label')
    return ['background-color: #ffeaea' if wrong else '' for _ in row]


In [ ]:
# Full results dataframe — sort by expected label then state
results_df = pd.read_csv('eval_results/testdata_full_scored.csv')
show_cols = [c for c in col_order if c in results_df.columns]
display_df = results_df[show_cols].sort_values(['expected_label', 'state'])


display(
    display_df.style
    .apply(highlight_wrong, axis=1)
    .format(precision=3)
)

In [ ]:
len(results_df)

## 4. Metrics on existing results

In [ ]:
valid = results_df[results_df['predicted_label'] != 'error'].copy()
y_true, y_pred = valid['expected_label'], valid['predicted_label']

acc = (y_true == y_pred).mean()
print(f'Files: {len(valid)}  |  Accuracy: {acc:.3f}\n')
print(classification_report(y_true, y_pred, labels=LABELS, zero_division=0))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred, labels=LABELS)
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(LABELS))); ax.set_xticklabels(LABELS, rotation=45, ha='right')
ax.set_yticks(range(len(LABELS))); ax.set_yticklabels(LABELS)
ax.set_xlabel('Predicted'); ax.set_ylabel('Expected')
ax.set_title('Confusion Matrix — existing results')
for i in range(len(LABELS)):
    for j in range(len(LABELS)):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                color='white' if cm[i, j] > cm.max()/2 else 'black')
plt.colorbar(im); plt.tight_layout(); plt.show()

In [ ]:
# Per-state breakdown
if 'state' in valid.columns:
    state_rows = []
    for state, grp in valid.groupby('state'):
        yt, yp = grp['expected_label'], grp['predicted_label']
        r = {'state': state, 'n': len(grp), 'accuracy': round((yt==yp).mean(), 3)}
        for cls in LABELS:
            if (yt==cls).any():
                r[f'{cls}_F1'] = round(f1_score(yt==cls, yp==cls, zero_division=0), 3)
        state_rows.append(r)
    state_df = pd.DataFrame(state_rows).set_index('state').sort_values('accuracy', ascending=False)
    display(state_df.style.background_gradient(cmap='RdYlGn', subset=['accuracy']))

## 4b. Manual-annotation evaluation — test_data_1

Loads the manual review CSV, derives ground-truth labels from the annotator columns,
scores every matched file in `test_data_1` with the current `CONFIG`, and compares.

**Label derivation from manual columns:**
- `manual_proper_document = FALSE` → `not_document`
- `manual_proper_document = TRUE` + `manual_image_clear_and_legible = FALSE` → `blur`
- both TRUE + `manual_is_full_landrecord = FALSE` → `cut`
- all TRUE → `accepted`

In [8]:
# ── Paths ────────────────────────────────────────────────────────────────────
MANUAL_CSV  = Path('/Users/mipl/Documents/ML_pipeline/Document_validation/manual_data - final_comparison_check_land_record_for_accuracy.csv')
TEST_DATA_1 = Path('/Users/mipl/Documents/ML_pipeline/Document_validation/test_data_1')
MANUAL_OUT  = OUT_DIR / 'manual_eval_scored.csv'

# ── Load manual annotations ──────────────────────────────────────────────────
ann = pd.read_csv(MANUAL_CSV)

def derive_manual_label(row):
    if not row['manual_proper_document']:
        return 'not_document'
    if not row['manual_image_clear_and_legible']:
        return 'blur'
    if not row['manual_is_full_landrecord']:
        return 'cut'
    return 'accepted'

ann['manual_label'] = ann.apply(derive_manual_label, axis=1)
print(f'Manual annotation CSV: {len(ann)} rows')
print('Ground-truth distribution:')
display(ann['manual_label'].value_counts().rename('count').to_frame())

# ── Build filename → (path, manual_label, metadata) map from test_data_1 ────
file_index = {}
for f in TEST_DATA_1.rglob('*'):
    if f.is_file() and f.suffix.lower() in EXTENSIONS:
        file_index[f.name] = f

matched = ann[ann['file_name'].isin(file_index)]
missing = ann[~ann['file_name'].isin(file_index)]
print(f'\nMatched {len(matched)}/{len(ann)} files in {TEST_DATA_1}')
if len(missing):
    print(f'Not found ({len(missing)}): {missing["file_name"].tolist()[:5]}')

Manual annotation CSV: 463 rows
Ground-truth distribution:


,count
manual_label,
blur,184
accepted,165
cut,100
not_document,14



Matched 463/463 files in /Users/mipl/Documents/ML_pipeline/Document_validation/test_data_1


In [9]:
# ── Score all matched files ───────────────────────────────────────────────────
# Pulls metadata columns from ann (state, doc_type, etc.) where available
meta_cols = [c for c in ['state', 'doc_type', 'manual_land_record', 'manual_phone_photo'] if c in ann.columns]

manual_rows = []
n = len(matched)
for i, (_, row) in enumerate(matched.iterrows()):
    fpath = file_index[row['file_name']]
    s = score_file(fpath, CONFIG)
    s['manual_label'] = row['manual_label']
    for col in meta_cols:
        s[col] = row[col]
    manual_rows.append(s)
    if (i + 1) % 50 == 0 or i == n - 1:
        print(f'  {i+1}/{n}', end='\r')

manual_df = pd.DataFrame(manual_rows)
manual_df.to_csv(MANUAL_OUT, index=False)
print(f'\nDone — {len(manual_df)} files → {MANUAL_OUT}')

  463/463
Done — 463 files → /Users/mipl/Documents/ML_pipeline/Document_validation/eval_results/manual_eval_scored.csv


In [10]:
# ── Display results — wrong rows highlighted red ──────────────────────────────
manual_col_order = [
    'filename', 'manual_label', 'predicted_label', 'raw_issues',
    'watermark_blur',
    'laplacian_variance', 'tenengrad', 'readability_contrast', 'grayscale_std',
    'blur_confidence', 'doc_confidence', 'cut_confidence', 'clear_confidence',
    'ink_ratio', 'edge_density', 'page_area_ratio',
    'file_path',
]
manual_show = [c for c in manual_col_order if c in manual_df.columns]

def highlight_wrong_manual(row):
    wrong = row.get('manual_label') != row.get('predicted_label')
    return ['background-color: #ffeaea' if wrong else '' for _ in row]

display_manual = manual_df[manual_show].sort_values(['manual_label', 'filename'])
display(
    display_manual.style
    .apply(highlight_wrong_manual, axis=1)
    .format(precision=3)
)


,filename,manual_label,predicted_label,raw_issues,laplacian_variance,tenengrad,readability_contrast,grayscale_std,blur_confidence,doc_confidence,cut_confidence,clear_confidence,ink_ratio,edge_density,page_area_ratio,file_path
169,100256_AgroFarm_44788_landrec.pdf,accepted,blur,"blur, not_clear, cut",307.660,48.180,35.510,48.570,0.197,0.757,1.000,0.250,0.066,0.045,0.469,/Users/mipl/Documents/ML_pipeline/Document_validation/test_data_1/Landrecord/Andhra_Pradesh/Unknown/phone_photo/ss/100256_AgroFarm_44788_landrec.pdf
454,117004_123250-ec0f396c-c82e-4894-a60e-f0c6d4b85b6e.pdf,accepted,accepted,,1812.300,82.930,38.200,20.740,0.000,0.983,0.550,1.000,0.067,0.056,0.988,/Users/mipl/Documents/ML_pipeline/Document_validation/test_data_1/Landrecord/WEST_BENGAL/phone_photo/normal/117004_123250-ec0f396c-c82e-4894-a60e-f0c6d4b85b6e.pdf
436,118067_124480-50e4d4eb-0480-4c38-8e6d-b7bbd8eb3d3c.pdf,accepted,accepted,,2443.550,77.180,50.090,20.670,0.000,0.971,0.000,1.000,0.046,0.045,0.987,/Users/mipl/Documents/ML_pipeline/Document_validation/test_data_1/ConcentForm/WEST_BENGAL/is_not_concent_form/118067_124480-50e4d4eb-0480-4c38-8e6d-b7bbd8eb3d3c.pdf
442,118071_124485-1e1b8f14-ad03-470f-8380-84a475f3d543.pdf,accepted,accepted,,2374.860,78.460,49.480,21.270,0.000,0.926,0.000,1.000,0.039,0.037,0.993,/Users/mipl/Documents/ML_pipeline/Document_validation/test_data_1/ConcentForm/WEST_BENGAL/is_not_concent_form/118071_124485-1e1b8f14-ad03-470f-8380-84a475f3d543.pdf
440,129215_135673-43549073-259c-4d8e-8fd7-40a598576c02.pdf,accepted,accepted,,2199.010,70.930,48.170,20.950,0.000,0.958,0.550,1.000,0.043,0.042,0.996,/Users/mipl/Documents/ML_pipeline/Document_validation/test_data_1/ConcentForm/WEST_BENGAL/is_not_concent_form/129215_135673-43549073-259c-4d8e-8fd7-40a598576c02.pdf
206,129996_136447-31397ba0-6914-48e8-a36e-28e4bccd7bdc.pdf,accepted,accepted,,4310.640,130.040,98.160,33.320,0.000,0.972,0.000,1.000,0.046,0.087,0.998,/Users/mipl/Documents/ML_pipeline/Document_validation/test_data_1/Landrecord/GUJRAT/phone_photo/ss/129996_136447-31397ba0-6914-48e8-a36e-28e4bccd7bdc.pdf
457,132664_137887-7873eeb3-3ccd-47c6-84c5-2cbcbc9190ad.pdf,accepted,accepted,,6455.580,134.300,147.190,67.840,0.000,0.909,0.000,1.000,0.120,0.094,0.958,/Users/mipl/Documents/ML_pipeline/Document_validation/test_data_1/Landrecord/WEST_BENGAL/phone_photo/ss/132664_137887-7873eeb3-3ccd-47c6-84c5-2cbcbc9190ad.pdf
459,133070_143756-ddcfb222-e174-4ffb-a96c-bd4c8c49af7d.pdf,accepted,blur,"blur, not_clear",536.840,57.370,37.840,23.330,0.044,0.936,0.550,0.250,0.043,0.034,0.993,/Users/mipl/Documents/ML_pipeline/Document_validation/test_data_1/Landrecord/WEST_BENGAL/phone_photo/normal/133070_143756-ddcfb222-e174-4ffb-a96c-bd4c8c49af7d.pdf
462,133315_143605-eaf8c051-e4db-4fab-b925-a675590013d2.pdf,accepted,not_document,"not_document, cut",949.100,53.460,45.600,43.670,0.109,0.642,0.961,1.000,0.050,0.038,0.207,/Users/mipl/Documents/ML_pipeline/Document_validation/test_data_1/Landrecord/WEST_BENGAL/phone_photo/normal/133315_143605-eaf8c051-e4db-4fab-b925-a675590013d2.pdf
458,133545_144844-14941535-4ec6-48c1-a8c8-ae80bfb55825.pdf,accepted,accepted,,3929.810,115.110,127.570,61.930,0.000,0.914,0.000,1.000,0.123,0.085,0.998,/Users/mipl/Documents/ML_pipeline/Document_validation/test_data_1/Landrecord/WEST_BENGAL/phone_photo/ss/133545_144844-14941535-4ec6-48c1-a8c8-ae80bfb55825.pdf


In [11]:
display_manual[(display_manual['manual_label'] == 'blur') & (display_manual['predicted_label'] == 'accepted')]

,filename,manual_label,predicted_label,raw_issues,laplacian_variance,tenengrad,readability_contrast,grayscale_std,blur_confidence,doc_confidence,cut_confidence,clear_confidence,ink_ratio,edge_density,page_area_ratio,file_path
266,119361_Farm_126428_landrec.pdf,blur,accepted,,2868.08,102.94,74.73,23.09,0.000,0.978,0.00,1.0,0.042,0.064,0.998,/Users/mipl/Documents/ML_pipeline/Document_val...
270,119758_Farm_126798_landrec.pdf,blur,accepted,,2943.16,109.44,52.79,24.67,0.000,0.959,0.00,1.0,0.040,0.079,0.998,/Users/mipl/Documents/ML_pipeline/Document_val...
448,128624_135125-31d18009-311d-49ef-ab13-51307902...,blur,accepted,,2027.50,106.11,67.96,29.54,0.000,0.962,0.55,1.0,0.086,0.064,0.993,/Users/mipl/Documents/ML_pipeline/Document_val...
452,132272_138478-9b2740c5-63fb-43f6-94ff-8a77bc7b...,blur,accepted,,1262.78,63.58,76.88,59.00,0.000,0.947,0.00,1.0,0.074,0.044,0.954,/Users/mipl/Documents/ML_pipeline/Document_val...
405,137632_147431-d46d2806-fc6b-49d0-be81-e16258a9...,blur,accepted,,779.99,67.39,49.61,29.47,0.133,0.933,0.00,1.0,0.058,0.034,0.998,/Users/mipl/Documents/ML_pipeline/Document_val...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,73773_ArrFarm_34116_landrec.pdf,blur,accepted,,989.20,87.89,43.47,44.77,0.000,0.898,0.55,1.0,0.126,0.102,0.954,/Users/mipl/Documents/ML_pipeline/Document_val...
115,75614_ArrFarm_34753_landrec.pdf,blur,accepted,,1832.36,120.00,96.44,51.40,0.000,0.952,0.00,1.0,0.079,0.081,0.969,/Users/mipl/Documents/ML_pipeline/Document_val...
167,75930_ArrFarm_34813_landrec.pdf,blur,accepted,,1969.23,97.84,54.20,33.99,0.000,0.953,0.00,1.0,0.076,0.082,0.990,/Users/mipl/Documents/ML_pipeline/Document_val...
23,90648_AgroFarm_40506_landrec.pdf,blur,accepted,,3320.80,125.67,74.54,38.08,0.000,0.967,0.00,1.0,0.054,0.086,0.998,/Users/mipl/Documents/ML_pipeline/Document_val...


In [ ]:
display_manual.columns

In [ ]:
display_manual[display_manual['filename'].str.startswith('119361_Farm_126428_landrec.pdf')]['file_path'].iloc[0]

In [ ]:
# ── Metrics on manual-annotated set ─────────────────────────────────────────
valid_m = manual_df[manual_df['predicted_label'] != 'error'].copy()
yt_m, yp_m = valid_m['manual_label'], valid_m['predicted_label']

acc_m = (yt_m == yp_m).mean()
print(f'Files: {len(valid_m)}  |  Accuracy: {acc_m:.3f}\n')
print(classification_report(yt_m, yp_m, labels=LABELS, zero_division=0))

# Confusion matrix
cm_m = confusion_matrix(yt_m, yp_m, labels=LABELS)
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm_m, cmap='Blues')
ax.set_xticks(range(len(LABELS))); ax.set_xticklabels(LABELS, rotation=45, ha='right')
ax.set_yticks(range(len(LABELS))); ax.set_yticklabels(LABELS)
ax.set_xlabel('Predicted'); ax.set_ylabel('Manual label')
ax.set_title('Confusion Matrix — manual annotation (test_data_1)')
for i in range(len(LABELS)):
    for j in range(len(LABELS)):
        ax.text(j, i, str(cm_m[i, j]), ha='center', va='center',
                color='white' if cm_m[i, j] > cm_m.max()/2 else 'black')
plt.colorbar(im); plt.tight_layout(); plt.show()

# Per-state breakdown (if state column exists)
if 'state' in valid_m.columns:
    state_rows_m = []
    for state, grp in valid_m.groupby('state'):
        yt, yp = grp['manual_label'], grp['predicted_label']
        r = {'state': state, 'n': len(grp), 'accuracy': round((yt==yp).mean(), 3)}
        for cls in LABELS:
            if (yt==cls).any():
                r[f'{cls}_F1'] = round(f1_score(yt==cls, yp==cls, zero_division=0), 3)
        state_rows_m.append(r)
    state_df_m = pd.DataFrame(state_rows_m).set_index('state').sort_values('accuracy', ascending=False)
    display(state_df_m.style.background_gradient(cmap='RdYlGn', subset=['accuracy']))

## 5. Re-run on a specific folder

Point `SCAN_FOLDER` to any directory.  
- If the folder follows the `doc_type/state/category/` layout, ground truth labels are inferred automatically.  
- Otherwise all files are scored with `expected_label = 'unknown'`.

Results are saved to `eval_results/folder_run_<folder_name>.csv`.

In [4]:
# ── Set the folder to scan ───────────────────────────────────────────────────
SCAN_FOLDER = Path('/Users/mipl/Documents/ML_pipeline/Document_validation/test_data/Landrecord/Andhra_Pradesh/old pathadhari/is_clear')   # ← change to any path

# ── Run ─────────────────────────────────────────────────────────────────────
assert SCAN_FOLDER.exists(), f'Folder not found: {SCAN_FOLDER}'

# Collect files and infer labels where possible
file_entries = []
for f in sorted(SCAN_FOLDER.rglob('*')):
    if not f.is_file() or f.suffix.lower() not in EXTENSIONS:
        continue
    # Try to infer label from parent folder name
    label = FOLDER_TO_LABEL.get(f.parent.name, 'unknown')
    file_entries.append({'file_path': f, 'expected_label': label})

print(f'Found {len(file_entries)} files in {SCAN_FOLDER}')
label_counts = pd.Series([e['expected_label'] for e in file_entries]).value_counts()
display(label_counts.rename('count').to_frame())

Found 4 files in /Users/mipl/Documents/ML_pipeline/Document_validation/test_data/Landrecord/Andhra_Pradesh/old pathadhari/is_clear


,count
accepted,4


In [5]:
# Score all files
folder_rows = []
n = len(file_entries)
for i, entry in enumerate(file_entries):
    s = score_file(entry['file_path'], CONFIG)
    s['expected_label'] = entry['expected_label']
    folder_rows.append(s)
    if (i + 1) % 50 == 0 or i == n - 1:
        print(f'  {i+1}/{n}', end='\r')

folder_df = pd.DataFrame(folder_rows)
out_csv = OUT_DIR / f'folder_run_{SCAN_FOLDER.name}.csv'
folder_df.to_csv(out_csv, index=False)
print(f'\nDone. {len(folder_df)} files → {out_csv}')

  4/4
Done. 4 files → /Users/mipl/Documents/ML_pipeline/Document_validation/eval_results/folder_run_is_clear.csv


In [6]:
# Display results
show_cols_f = [c for c in col_order if c in folder_df.columns]
folder_display = folder_df[show_cols_f].sort_values(['expected_label', 'filename'] if 'expected_label' in folder_df.columns else ['filename'])

display(
    folder_display.style
    .apply(highlight_wrong, axis=1)
    .format(precision=3)
)

,filename,expected_label,predicted_label,raw_issues,laplacian_variance,tenengrad,readability_contrast,grayscale_std,blur_confidence,doc_confidence,cut_confidence,clear_confidence,ink_ratio,edge_density,page_area_ratio,thresh_laplacian,thresh_tenengrad,thresh_min_doc_conf,thresh_min_reject,file_path
0,45569_ArrFarm_20213_landrec.pdf,accepted,accepted,,3064.520,113.770,62.140,31.140,0.000,0.935,0.550,1.000,0.104,0.079,0.954,900.000,60.000,0.750,0.750,/Users/mipl/Documents/ML_pipeline/Document_validation/test_data/Landrecord/Andhra_Pradesh/old pathadhari/is_clear/45569_ArrFarm_20213_landrec.pdf
1,72659_ArrFarm_33035_landrec.pdf,accepted,accepted,,2848.050,73.730,130.180,44.740,0.000,0.974,0.000,1.000,0.074,0.060,0.998,900.000,60.000,0.750,0.750,/Users/mipl/Documents/ML_pipeline/Document_validation/test_data/Landrecord/Andhra_Pradesh/old pathadhari/is_clear/72659_ArrFarm_33035_landrec.pdf
2,73888_ArrFarm_34360_landrec.pdf,accepted,accepted,,1519.290,81.240,72.590,63.910,0.000,0.865,0.450,1.000,0.067,0.077,0.766,900.000,60.000,0.750,0.750,/Users/mipl/Documents/ML_pipeline/Document_validation/test_data/Landrecord/Andhra_Pradesh/old pathadhari/is_clear/73888_ArrFarm_34360_landrec.pdf
3,75604_ArrFarm_34748_landrec.pdf,accepted,cut,cut,856.850,82.270,65.210,34.260,0.048,0.943,1.000,1.000,0.066,0.054,0.890,900.000,60.000,0.750,0.750,/Users/mipl/Documents/ML_pipeline/Document_validation/test_data/Landrecord/Andhra_Pradesh/old pathadhari/is_clear/75604_ArrFarm_34748_landrec.pdf


In [ ]:
# Metrics — only if ground truth labels are available
known = folder_df[folder_df['expected_label'] != 'unknown']
if len(known) == 0:
    print('No ground-truth labels found — cannot compute metrics.')
    print('Prediction distribution:')
    display(folder_df['predicted_label'].value_counts().rename('count').to_frame())
else:
    valid_f = known[known['predicted_label'] != 'error']
    yt, yp = valid_f['expected_label'], valid_f['predicted_label']
    print(f'Files with labels: {len(valid_f)}  |  Accuracy: {(yt==yp).mean():.3f}\n')
    print(classification_report(yt, yp, labels=LABELS, zero_division=0))

    cm = confusion_matrix(yt, yp, labels=LABELS)
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(cm, cmap='Blues')
    ax.set_xticks(range(len(LABELS))); ax.set_xticklabels(LABELS, rotation=45, ha='right')
    ax.set_yticks(range(len(LABELS))); ax.set_yticklabels(LABELS)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Expected')
    ax.set_title(f'Confusion Matrix — {SCAN_FOLDER.name}')
    for i in range(len(LABELS)):
        for j in range(len(LABELS)):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                    color='white' if cm[i, j] > cm.max()/2 else 'black')
    plt.colorbar(im); plt.tight_layout(); plt.show()

## 6. Feature score distributions (optional)

Shows how Laplacian variance, Tenengrad, and document confidence separate by class — useful for threshold tuning.

In [7]:
# Use whichever df has feature scores
plot_df = results_df if 'laplacian_variance' in results_df.columns else folder_df
plot_df = plot_df[plot_df['expected_label'].isin(LABELS)].copy()

score_cols = ['laplacian_variance', 'tenengrad', 'blur_confidence', 'doc_confidence',
              'cut_confidence', 'clear_confidence']
score_cols = [c for c in score_cols if c in plot_df.columns]

COLORS = {'accepted': '#2ecc71', 'blur': '#e74c3c', 'cut': '#f39c12', 'not_document': '#9b59b6'}

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for ax, col in zip(axes, score_cols):
    for label in LABELS:
        vals = plot_df[plot_df['expected_label'] == label][col].dropna()
        if len(vals):
            ax.hist(vals, bins=30, alpha=0.5, label=label, color=COLORS.get(label, 'gray'))
    ax.set_title(col)
    ax.set_xlabel('value'); ax.set_ylabel('count')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

# Hide unused axes
for ax in axes[len(score_cols):]:
    ax.set_visible(False)

plt.suptitle('Feature score distributions by class', fontsize=13)
plt.tight_layout()
plt.show()

NameError: name 'results_df' is not defined

In [ ]:
import cv2
import numpy as np

# 1. Load image in grayscale
img = cv2.imread('/Users/mipl/Documents/Screenshot 2026-06-18 at 11.53.41 AM.png', cv2.IMREAD_GRAYSCALE)

# 2. Compute Sobel gradients (First Derivatives)
sobel_x = cv2.Sobel(img, cv2.CV_64F, 1, 0, ksize=3)
sobel_y = cv2.Sobel(img, cv2.CV_64F, 0, 1, ksize=3)

# 3. Calculate Squared Gradient Magnitude
grad_magnitude_sq = (sobel_x ** 2) + (sobel_y ** 2)

# 4. Take the square root to get the actual magnitude map
grad_magnitude = np.sqrt(grad_magnitude_sq)

# 5. Normalize to 0-255 so it can be rendered as a visual image
tenengrad_visual = cv2.normalize(grad_magnitude, None, 0, 255, cv2.NORM_MINMAX, dtype=cv2.CV_8U)

# 6. Calculate the final score (sum of all pixels)
tenengrad_score = np.sum(grad_magnitude_sq)
print(f"Tenengrad Sharpness Score: {tenengrad_score}")

# Display the visualization
# cv2.imshow("What Tenengrad Sees", tenengrad_visual)
# cv2.waitKey(0)
# cv2.destroyAllWindows()